<a href="https://colab.research.google.com/github/Shibin2000/weather-etl-pipeline/blob/main/weather_etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Weather ETL Pipeline

This project extracts real-time weather data from the OpenWeatherMap API,
transforms it using Python and Pandas, loads it into a DuckDB warehouse,
and visualizes the results using Plotly.

### Pipeline Steps
1. Extract — Fetch live weather data from API
2. Transform — Clean and categorize the data
3. Load — Store in DuckDB warehouse
4. Analyze — Run SQL queries
5. Visualize — Generate interactive charts

In [28]:
# Installing required libraries for the pipeline
!pip install duckdb plotly requests pandas

##  Extract
Fetching live weather data for 5 US cities using the OpenWeatherMap API.

In [29]:
import requests
import pandas as pd
from datetime import datetime

# My API key from OpenWeatherMap
API_KEY = "275ac1a9e2a70109224116030bf590bf"

# Cities I want to track
cities = ["Houston", "Dallas", "Austin", "New York", "Chicago"]

# Fetching weather data for each city
weather_data = []
for city in cities:
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=imperial"
    response = requests.get(url)
    data = response.json()

    weather_data.append({
        'city': city,
        'temperature_f': data['main']['temp'],
        'humidity': data['main']['humidity'],
        'weather': data['weather'][0]['description'],
        'wind_speed': data['wind']['speed']
    })
    print(f"Fetched: {city} → {data['main']['temp']}°F")

df = pd.DataFrame(weather_data)
print("\nRaw data extracted successfully!")
print(df)

Fetched: Houston → 56.61°F
Fetched: Dallas → 61.36°F
Fetched: Austin → 61.95°F
Fetched: New York → 34.65°F
Fetched: Chicago → 24.21°F

Raw data extracted successfully!
       city  temperature_f  humidity          weather  wind_speed
0   Houston          56.61        35        clear sky       18.41
1    Dallas          61.36        24    broken clouds       12.64
2    Austin          61.95        18    broken clouds       11.30
3  New York          34.65        37  overcast clouds       15.03
4   Chicago          24.21        51  overcast clouds        8.46


##  Transform
Cleaning the raw data and adding a temperature category column.

In [30]:
# Cleaning and transforming the raw weather data

df['recorded_at'] = datetime.now()

# Categorizing temperature into human readable labels
df['temp_category'] = df['temperature_f'].apply(
    lambda x: 'Hot' if x > 80
    else ('Warm' if x > 60 else 'Cold')
)

# Rounding temperature for cleaner output
df['temperature_f'] = df['temperature_f'].round(2)

print("Data after transformation:")
print(df[['city', 'temperature_f', 'temp_category', 'humidity', 'weather']])

Data after transformation:
       city  temperature_f temp_category  humidity          weather
0   Houston          56.61          Cold        35        clear sky
1    Dallas          61.36          Warm        24    broken clouds
2    Austin          61.95          Warm        18    broken clouds
3  New York          34.65          Cold        37  overcast clouds
4   Chicago          24.21          Cold        51  overcast clouds


##  Load
Loading the transformed data into a DuckDB local warehouse.

In [31]:
import duckdb

# Creating our local data warehouse using DuckDB
conn = duckdb.connect('weather_warehouse.db')

# Drop table if it exists to ensure schema is updated
conn.execute("DROP TABLE IF EXISTS weather_data")

# Create table if it doesn't exist
conn.execute("""
    CREATE TABLE weather_data (
        city VARCHAR,
        temperature_f FLOAT,
        humidity INT,
        weather VARCHAR,
        wind_speed FLOAT,
        temp_category VARCHAR,
        recorded_at TIMESTAMP
    )
""")

# Loading transformed data into warehouse
conn.execute("""
    INSERT INTO weather_data
    SELECT city, temperature_f, humidity, weather,
           wind_speed, temp_category, recorded_at FROM df
""")

total = conn.execute("SELECT COUNT(*) FROM weather_data").fetchone()[0]
print(f"Data loaded successfully! Total records: {total}")
conn.close()

Data loaded successfully! Total records: 5


##  SQL Analysis
Running SQL queries to find the hottest city, average temperature,
and temperature distribution.

In [32]:
import duckdb

conn = duckdb.connect('weather_warehouse.db')

# Which city is hottest right now?
print("=== Hottest City ===")
print(conn.execute("""
    SELECT city, temperature_f
    FROM weather_data
    ORDER BY temperature_f DESC
    LIMIT 1
""").fetchdf())

# Average temperature across all cities
print("\n=== Average Temperature ===")
print(conn.execute("""
    SELECT ROUND(AVG(temperature_f), 2) as avg_temp_f
    FROM weather_data
""").fetchdf())

# How many cities are hot, warm, cold?
print("\n=== Temperature Categories ===")
print(conn.execute("""
    SELECT temp_category, COUNT(*) as num_cities
    FROM weather_data
    GROUP BY temp_category
    ORDER BY num_cities DESC
""").fetchdf())

conn.close()

=== Hottest City ===
     city  temperature_f
0  Austin      61.950001

=== Average Temperature ===
   avg_temp_f
0       47.76

=== Temperature Categories ===
  temp_category  num_cities
0          Cold           3
1          Warm           2


# Temperature Dashboard
Visualizing temperature across all cities using an interactive bar chart.

In [33]:
import plotly.express as px
import duckdb

conn = duckdb.connect('weather_warehouse.db')
df_viz = conn.execute("SELECT * FROM weather_data").fetchdf()
conn.close()

# Bar chart showing temperature by city
fig = px.bar(
    df_viz,
    x='city',
    y='temperature_f',
    color='temperature_f',
    title='Live Weather Temperatures Across US Cities',
    labels={'temperature_f': 'Temperature (°F)', 'city': 'City'},
    color_continuous_scale='RdYlBu_r',
    text='temperature_f'
)

fig.update_traces(texttemplate='%{text:.1f}°F', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()
print("Dashboard generated!")

Dashboard generated!


##  Airflow DAG Structure
Showing how this pipeline would be scheduled in production using Apache Airflow.

In [34]:
# Airflow DAG

from datetime import datetime, timedelta

dag_config = {
    'dag_id': 'weather_etl_pipeline',
    'description': 'Fetch, transform and load weather data every hour',
    'schedule_interval': timedelta(hours=1),
    'start_date': datetime(2026, 3, 17),
    'tasks': {
        'extract': 'Fetch weather from OpenWeatherMap API',
        'transform': 'Clean data and add temperature categories',
        'load': 'Store in DuckDB warehouse',
        'visualize': 'Generate Plotly dashboard'
    }
}

print("DAG Configuration:")
print(f"Pipeline: {dag_config['dag_id']}")
print(f"Schedule: Every 1 hour")
print(f"\nTask Order:")
for i, (task, desc) in enumerate(dag_config['tasks'].items(), 1):
    print(f"  {i}. {task} → {desc}")

DAG Configuration:
Pipeline: weather_etl_pipeline
Schedule: Every 1 hour

Task Order:
  1. extract → Fetch weather from OpenWeatherMap API
  2. transform → Clean data and add temperature categories
  3. load → Store in DuckDB warehouse
  4. visualize → Generate Plotly dashboard


##  Data Quality Checks
Validating the data to ensure no missing values, duplicates, or invalid ranges.

In [35]:
# Data quality checks

print("Running data quality checks...")
print("-" * 40)

missing = df.isnull().sum().sum()
print(f"Missing values: {missing} ✓" if missing == 0 else f"Missing values: {missing} ✗")

valid_temp = df['temperature_f'].between(-50, 130).all()
print(f"Temperature range valid: {'✓' if valid_temp else '✗'}")

print(f"Cities loaded: {len(df)} out of 5 ✓")

duplicates = df['city'].duplicated().sum()
print(f"Duplicate cities: {duplicates} ✓" if duplicates == 0 else f"Duplicates found: {duplicates} ✗")

print("-" * 40)
print("All quality checks passed!")

Running data quality checks...
----------------------------------------
Missing values: 0 ✓
Temperature range valid: ✓
Cities loaded: 5 out of 5 ✓
Duplicate cities: 0 ✓
----------------------------------------
All quality checks passed!


##  Humidity Dashboard
Comparing humidity levels across all cities.

In [36]:
# Humidity comparison across cities
fig2 = px.bar(
    df_viz,
    x='city',
    y='humidity',
    color='humidity',
    title='Humidity Levels Across US Cities',
    labels={'humidity': 'Humidity (%)', 'city': 'City'},
    color_continuous_scale='Blues',
    text='humidity'
)
fig2.update_traces(texttemplate='%{text}%', textposition='outside')
fig2.show()
print("Humidity chart done!")

Humidity chart done!
